# 第 1 周 · 第 2 天 —— 一种 API 形状，多种后端

OpenAI 的 Chat Completions 格式几乎成了事实标准，因此**同一种请求形状**可以对接很不一样的后端。本笔记本用四种方式发出**相同结构**的调用：

1. **裸 HTTP** —— SDK 底层实际在做的事  
2. **官方 OpenAI SDK** —— 更方便的封装  
3. **Google Gemini** —— 走其 OpenAI 兼容端点  
4. **Ollama 本地模型** —— 同一形状，跑在你自己的机器上  

各供应商之间通常只换三样东西：**base URL**、**API key**、**model 名**。

> 需要 `OPENAI_API_KEY`（Gemini 那格还要 `GEMINI_API_KEY`）。Ollama 相关单元格需要本机已启动 [Ollama](https://ollama.com) 服务。


In [ ]:
# ========== 导入 + 加载并校验 OpenAI Key ==========

# os：读环境变量
import os

# requests：后面用裸 HTTP POST 调 Chat Completions
import requests
# load_dotenv：从 .env 读密钥
from dotenv import load_dotenv
# 笔记本里渲染 Markdown
from IPython.display import Markdown, display

# 加载 .env（此处未传 override，保持原调用）
load_dotenv()
# 取出 OpenAI 密钥，后面既给 HTTP Bearer，也给 SDK
key = os.getenv("OPENAI_API_KEY")
# 缺 key 立刻失败，避免无效请求
if not key:
    raise ValueError("OPENAI_API_KEY not found in environment variables.")


## 1. 直接用 HTTP 调 API

在用 SDK 之前，值得先看一眼**裸请求**：Chat Completions 端点就是一次带 Bearer token 的 `POST` + JSON body。这里的鉴权与载荷形状，正是下一节 SDK 替你组装的内容。


In [ ]:
# ========== 组装 HTTP 头与 JSON 载荷 ==========

# Content-Type 声明 JSON；Authorization 用 Bearer + 密钥
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {key}",
}

# 与 Chat Completions 约定一致的 body：model + messages
payload = {
    "model": "gpt-5-nano",
    # 发给模型的 user 内容保留英文
    "messages": [{"role": "user", "content": "Tell me a fun fact."}],
}


In [ ]:
# ========== 裸 POST：自己从 JSON 字典里抠出回复文本 ==========

# 端点返回普通 JSON；用字典下标取出 generated text（SDK 下一格会帮你解包）
response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload).json()
# choices[0].message.content 对应 SDK 里的同名属性路径
print(response["choices"][0]["message"]["content"])


## 2. 使用 OpenAI Python SDK

SDK 封装了同一次 HTTP 调用：自动组 header、发请求，并返回带类型的对象，于是可以写 `response.choices[0].message.content`，而不必手抠原始 dict。


In [ ]:
# ========== 官方 SDK：同一请求，更短的写法 ==========

# 导入 OpenAI 客户端类
from openai import OpenAI

# 显式传入刚才读到的 key（与依赖环境变量自动发现等价）
client = OpenAI(api_key=key)
# chat.completions.create：与裸 HTTP 对应的高层 API
response = client.chat.completions.create(
    model="gpt-5-nano",
    # prompt 字符串保持英文
    messages=[{"role": "user", "content": "Tell me a fun fact about India."}],
)

# 用 Markdown 展示助手回复
display(Markdown(response.choices[0].message.content))


## 3. 同一套 SDK，换供应商 —— Google Gemini

共享 API 形状的收益在这里：把 OpenAI 客户端指到 Gemini 的 **OpenAI 兼容 base URL**，换 key 和 model，**其余调用一字不改**。不必学新 SDK。


In [ ]:
# ========== Gemini：只换 base_url / api_key / model ==========

# 只有这三处不同——下面的 create 调用与 OpenAI 那格同形，因为 Gemini 讲 OpenAI 线格式
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=os.getenv("GEMINI_API_KEY"))
response = gemini.chat.completions.create(
    # Gemini 侧的模型 id（保留原字符串）
    model="gemini-3.1-flash-lite",
    messages=[{"role": "user", "content": "Tell me a fun fact about India."}],
)

display(Markdown(response.choices[0].message.content))


## 4. 用 Ollama 在本地跑

同样的技巧，后端换成你自己的机器：先 pull 模型，再把客户端指到 Ollama 本地的 OpenAI 兼容服务。适合离线、隐私敏感、以及零按 token 计费的实验。


In [ ]:
# ========== 拉取一个轻量本地模型 ==========

# shell 魔法：从 Ollama 仓库下载 llama3.2:1b（体积小，适合演示）
!ollama pull llama3.2:1b


In [ ]:
# ========== Ollama：本地 OpenAI 兼容端点 ==========

# Ollama 要求传 api_key，但通常忽略其值，任意非空占位即可；本地服务讲同一套 Chat Completions
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
response = ollama.chat.completions.create(
    # 注意：这里用的是 llama3.2:3b（与上一格 pull 的 :1b 可能不同；需本机已有该标签）
    model="llama3.2:3b",
    messages=[{"role": "user", "content": "Tell me a fun fact about India."}],
)

display(Markdown(response.choices[0].message.content))
